In [2]:
!pip install -q pandas openpyxl scikit-learn gradio

In [3]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

filename = next(iter(uploaded))

df = pd.read_excel(filename)

print("Dataset uploaded successfully!")
print("Dataset shape:", df.shape)

display(df.head())
print(df.columns.tolist())

Saving colour_prediction_dataset.xlsx to colour_prediction_dataset.xlsx
Dataset uploaded successfully!
Dataset shape: (30000, 5)


,Red,Green,Blue,Color,HEX
0,255,0,29,Red,#FF001D
1,255,0,0,Red,#FF0000
2,255,34,0,Red,#FF2200
3,255,0,0,Red,#FF0000
4,255,0,0,Red,#FF0000


['Red', 'Green', 'Blue', 'Color', 'HEX']


In [4]:
import numpy as np

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Remove missing values
df = df.dropna(subset=["Red", "Green", "Blue", "Color"])

# Input features
X = df[["Red", "Green", "Blue"]].astype(float)

# Output target
y = df["Color"].astype(str)

# Convert colour names into numbers
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("Preprocessing completed!")
print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)
print("Colour categories:", list(encoder.classes_))

Preprocessing completed!
Training data: (24000, 3)
Testing data: (6000, 3)
Colour categories: ['Black', 'Blue', 'Brown', 'Green', 'Orange', 'Pink', 'Purple', 'Red', 'White', 'Yellow']


In [5]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline

# Create ANN model
model = Pipeline([
    ("scaler", StandardScaler()),

    ("ann", MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        solver="adam",
        learning_rate_init=0.001,
        max_iter=300,
        early_stopping=False,
        random_state=42
    ))
])

# Train model
model.fit(X_train, y_train)

# Predict testing data
y_pred = model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print("ANN MODEL TRAINED SUCCESSFULLY!")
print("Model Accuracy:", round(accuracy * 100, 2), "%")

print("\nClassification Report:")

print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

ANN MODEL TRAINED SUCCESSFULLY!
Model Accuracy: 89.37 %

Classification Report:
              precision    recall  f1-score   support

       Black       0.95      0.94      0.95       600
        Blue       0.97      0.99      0.98       600
       Brown       0.77      0.78      0.78       600
       Green       0.98      0.99      0.98       600
      Orange       0.76      0.80      0.78       600
        Pink       0.92      0.90      0.91       600
      Purple       0.87      0.83      0.85       600
         Red       0.89      0.88      0.89       600
       White       0.95      0.98      0.97       600
      Yellow       0.87      0.84      0.85       600

    accuracy                           0.89      6000
   macro avg       0.89      0.89      0.89      6000
weighted avg       0.89      0.89      0.89      6000



In [6]:
import re

def predict_color(hex_code):

    hex_code = hex_code.strip()

    if not re.fullmatch(r"#?[0-9A-Fa-f]{6}", hex_code):
        return "Invalid HEX code! Example: #FF0000"

    hex_code = hex_code.lstrip("#")

    # HEX to RGB
    r = int(hex_code[0:2], 16)
    g = int(hex_code[2:4], 16)
    b = int(hex_code[4:6], 16)

    # ANN prediction
    prediction = model.predict([[r, g, b]])[0]

    # Convert numerical output to colour name
    color_name = encoder.inverse_transform([prediction])[0]

    return {
        "HEX Code": "#" + hex_code.upper(),
        "RGB Values": (r, g, b),
        "Predicted Colour": color_name
    }


# Test prediction
print(predict_color("#FF0000"))
print(predict_color("#00FF00"))
print(predict_color("#0000FF"))

{'HEX Code': '#FF0000', 'RGB Values': (255, 0, 0), 'Predicted Colour': 'Red'}
{'HEX Code': '#00FF00', 'RGB Values': (0, 255, 0), 'Predicted Colour': 'Green'}
{'HEX Code': '#0000FF', 'RGB Values': (0, 0, 255), 'Predicted Colour': 'Blue'}


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [7]:
import gradio as gr

def colour_app(hex_code):

    result = predict_color(hex_code)

    if isinstance(result, str):
        return result, ""

    color_name = result["Predicted Colour"]
    rgb = result["RGB Values"]
    hex_value = result["HEX Code"]

    output = f"""
    PREDICTED COLOUR: {color_name}

    HEX CODE: {hex_value}

    RGB VALUES: {rgb}
    """

    preview = f"""
    <div style="
        background:{hex_value};
        height:150px;
        border-radius:15px;
        border:2px solid gray;
        margin:15px;">
    </div>
    """

    return output, preview


with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("# AI Colour Prediction Using ANN")

    gr.Markdown("Enter a HEX colour code to predict its category.")

    hex_input = gr.Textbox(
        label="HEX Code",
        placeholder="#FF0000"
    )

    button = gr.Button("Predict Colour")

    result = gr.Textbox(label="Prediction Result")

    preview = gr.HTML(label="Colour Preview")

    button.click(
        colour_app,
        inputs=hex_input,
        outputs=[result, preview]
    )

demo.launch(share=True)

/tmp/ipykernel_828/3922794836.py:35: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://207e83f59d8f45300b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import joblib

joblib.dump(model, "ann_colour_model.pkl")
joblib.dump(encoder, "colour_encoder.pkl")

print("ANN model and encoder saved!")